In [1]:
!pip install -U langchain langchain-community langchain-core chromadb pypdf

In [2]:
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-

In [3]:
import langchain
print(langchain.__version__)

1.2.7


Load text documents

In [4]:
loader = TextLoader("data.txt")
docs = loader.load()

Split documents

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)

len(splits)



1

Create embeddings (LOCAL)

In [6]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_10536\1863499803.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


Vector store (Chroma)

In [7]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)


Retriever

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


LLM

In [9]:
llm = ChatOllama(
    model="llama3",
    temperature=0
)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_10536\1299597105.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


RAG prompt

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}
""")


Build RAG chain (Runnable-style)

In [11]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


Ask questions

In [12]:
rag_chain.invoke("What is LangChain?")


'According to the context, LangChain is a framework for building applications with large language models. It helps with prompting, memory, retrieval, agents, and more.'

Hallucination test

In [13]:
rag_chain.invoke("Who is the president of Mars?")


"I don't know."

Debug retrieval

In [14]:
retrieved_docs = retriever.invoke("What is LangChain?")

for doc in retrieved_docs:
    print("----")
    print(doc.page_content)


----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
